<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-06 14:26:56
=================== PORTFOLIO QUALITY ====================
qualified stocks: 86
with latest results: 86
still star stocks: 46


=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.53 C
CAGR/XIRR %:     1.01%
Est FTT Value:   2.74 C
Future Upside:   1.22 C (79.64%)
=================== PERFORMANCE ==========================
Current PnL:             -43.91 L (-22.88%)
Total Booked PnL:        48.59 L (32.6%)
Booked + Current PnL:    4.68 L (3.14%)
CY Booked PnL:           5.71 L (3.74%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               -17.39 K (-0.11%)
CY Investment:            1.92 C
Reserve:                  1.90 K
=================== P&L DETAIL ============================
Total Profit:             2.24 L
Total Loss:               -46.15 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
85,ZYDUSLIFE,0.34,26.23,7.35,35.51,19443.0,54963.0,264537.0,1285.01,24.82,56.0,H-LC,11.85,120.0,2.83,1.73,39.21,AR,ATH,PHARMA
76,TTKPRESTIG,9.36,-21.67,27.78,0.09,21930.0,-21836.0,78941.0,770.00,-4.67,72.0,M-SC,3.53,250.0,-1.00,0.52,42.08,OX40N,NTT,DURABLES
40,ICICIPRULI,1.09,-29.23,22.14,-13.56,30866.0,-57591.0,139414.0,522.35,-54.58,44.0,X-MC,9.52,78.0,-1.87,0.91,5.07,X40,ATH,INSURANCE
12,BOSCH-HCIL,0.20,-26.25,35.58,-0.01,33537.0,-33546.0,94259.0,2282.00,733.99,43.0,M-SC,19.87,231.0,-1.00,0.62,60.44,OX40N,BTT,AC
66,SIS,-1.53,-11.91,37.66,21.26,36833.0,-13227.0,97805.0,528.00,88.01,45.0,H-SC,7.90,168.0,-0.36,0.64,45.42,OX40N,NTT,MISC


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
52,MARUTI,1.15,-6.50,33.78,25.08,95102.0,-19584.0,281534.0,17120.06,-12.21,72.0,X-LC,12.93,33.0,-0.21,1.84,14.57,XR,ATH,AUTO
38,HYUNDAI,2.91,12.90,27.49,43.94,78003.0,32419.0,283751.0,2782.73,16.88,70.0,X-LC,5.01,29.0,0.42,1.86,30.28,X40N,ATH,AUTO
23,DMART,0.06,-15.95,57.56,32.43,171359.0,-56495.0,297705.0,5391.45,-30.04,52.0,X-LC,16.35,14.0,-0.33,1.95,9.19,X40N,ATH,FMCG
35,HDFCLIFE,2.32,-3.62,45.83,40.56,137444.0,-11259.0,299900.0,811.41,-29.81,62.0,X-LC,9.83,31.0,-0.08,1.96,8.33,X40,ATH,INSURANCE
4,ASIANPAINT,-0.04,-8.54,35.65,24.06,107930.0,-28284.0,302748.0,3422.40,-19.82,67.0,X-LC,5.86,30.0,-0.26,1.98,20.25,X40,ATH,PAINTS


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
76,TTKPRESTIG,9.36,-21.67,27.78,0.09,21930.0,-21836.0,78941.0,770.00,-4.67,72.0,M-SC,3.53,250.0,-1.00,0.52,42.08,OX40N,NTT,DURABLES
51,M&M,4.21,1.75,15.46,17.48,47362.0,5280.0,306355.0,3762.88,-11.41,75.0,X-LC,8.45,25.0,0.11,2.01,18.46,X40N,ATH,AUTO
67,STARHEALTH,3.97,9.12,34.95,47.26,107213.0,25639.0,306762.0,761.00,-3.79,59.0,H-SC,4.80,173.0,0.24,2.01,32.64,XY24,NTT,INSURANCE
32,HAVELLS,3.33,-29.33,85.98,31.42,199282.0,-96214.0,231777.0,2052.62,-39.10,58.0,X-MC,13.00,63.0,-0.48,1.52,9.44,X40,ATH,ELECTRICAL
38,HYUNDAI,2.91,12.90,27.49,43.94,78003.0,32419.0,283751.0,2782.73,16.88,70.0,X-LC,5.01,29.0,0.42,1.86,30.28,X40N,ATH,AUTO


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-7.26,37.94,49.66,106.44,101139.0,56018.0,203662.0,1629.94,0.43,34.0,M-SC,24.99,220.0,0.55,1.33,54.31,OX40N,ATH,CABLES
80,VALIANTORG,-6.70,-28.39,143.18,74.15,167755.0,-46441.0,117164.0,838.00,NaN,34.0,NaN,43.91,145.0,-0.28,0.77,74.36,XR,NTT,CHEMICALS
11,BERGEPAINT,-5.50,-12.06,42.50,25.32,96335.0,-31079.0,226670.0,680.00,-23.45,52.0,X-MC,2.04,68.0,-0.32,1.48,18.86,XY24,NTT,PAINTS
49,KANSAINER,-4.12,-34.65,73.63,13.47,129764.0,-93429.0,176238.0,340.00,-31.54,59.0,H-SC,7.23,162.0,-0.72,1.15,24.57,XY24,NTT,PAINTS
21,DEN,-3.94,-51.47,192.74,42.07,79008.0,-43472.0,40992.0,75.00,NaN,50.0,NaN,6.06,235.0,-0.55,0.27,11.49,AR,NTT,ENTERTAINMENT


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-7.26,37.94,49.66,106.44,101139.0,56018.0,203662.0,1629.94,0.43,34.0,M-SC,24.99,220.0,0.55,1.33,54.31,OX40N,ATH,CABLES
66,SIS,-1.53,-11.91,37.66,21.26,36833.0,-13227.0,97805.0,528.00,88.01,45.0,H-SC,7.90,168.0,-0.36,0.64,45.42,OX40N,NTT,MISC
76,TTKPRESTIG,9.36,-21.67,27.78,0.09,21930.0,-21836.0,78941.0,770.00,-4.67,72.0,M-SC,3.53,250.0,-1.00,0.52,42.08,OX40N,NTT,DURABLES
12,BOSCH-HCIL,0.20,-26.25,35.58,-0.01,33537.0,-33546.0,94259.0,2282.00,733.99,43.0,M-SC,19.87,231.0,-1.00,0.62,60.44,OX40N,BTT,AC
49,KANSAINER,-4.12,-34.65,73.63,13.47,129764.0,-93429.0,176238.0,340.00,-31.54,59.0,H-SC,7.23,162.0,-0.72,1.15,24.57,XY24,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-7.26,37.94,49.66,106.44,101139.0,56018.0,203662.0,1629.94,0.43,34.0,M-SC,24.99,220.0,0.55,1.33,54.31,OX40N,ATH,CABLES
85,ZYDUSLIFE,0.34,26.23,7.35,35.51,19443.0,54963.0,264537.0,1285.01,24.82,56.0,H-LC,11.85,120.0,2.83,1.73,39.21,AR,ATH,PHARMA


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-7.26,37.94,49.66,106.44,101139.0,56018.0,203662.0,1629.94,0.43,34.0,M-SC,24.99,220.0,0.55,1.33,54.31,OX40N,ATH,CABLES
67,STARHEALTH,3.97,9.12,34.95,47.26,107213.0,25639.0,306762.0,761.00,-3.79,59.0,H-SC,4.80,173.0,0.24,2.01,32.64,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,-7.26,37.94,49.66,106.44,101139.0,56018.0,203662.0,1629.94,0.43,34.0,M-SC,24.99,220.0,0.55,1.33,54.31,OX40N,ATH,CABLES
84,WIPRO,-0.04,-8.86,111.77,93.01,254925.0,-22160.0,228080.0,420.00,-23.12,69.0,M-LC,7.84,98.0,-0.09,1.49,26.49,XR,NTT,IT
70,TANLA,-0.63,-26.36,192.19,115.17,383208.0,-71371.0,199390.0,1903.93,NaN,74.0,NaN,14.19,147.0,-0.19,1.31,76.26,AR,ATH,IT
42,INDIAMART,-1.75,-33.45,201.89,100.92,165723.0,-41250.0,82086.0,4675.60,NaN,44.0,NaN,16.76,140.0,-0.25,0.54,4.14,AR,ATH,MISC
3,ALKYLAMINE,0.32,-33.01,187.40,92.54,126755.0,-33324.0,67639.0,4520.85,NaN,34.0,NaN,8.12,142.0,-0.26,0.44,28.44,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
37,HONAUT,-0.86,-22.06,93.59,50.88,140666.0,-42538.0,150300.0,58192.80,-39.13,36.0,X-SC,6.99,90.0,-0.30,0.98,14.51,X40N,ATH,ELECTRICAL
6,AWL,-0.74,-42.65,191.20,67.02,390731.0,-151952.0,204357.0,485.00,-57.79,38.0,X-SC,7.91,81.0,-0.39,1.34,1.65,XY24,NTT,FMCG
40,ICICIPRULI,1.09,-29.23,22.14,-13.56,30866.0,-57591.0,139414.0,522.35,-54.58,44.0,X-MC,9.52,78.0,-1.87,0.91,5.07,X40,ATH,INSURANCE
27,GLAXO,1.50,1.88,33.19,35.69,67384.0,3740.0,203024.0,3380.16,-23.53,46.0,X-MC,9.34,56.0,0.06,1.33,20.66,X40,ATH,PHARMA
56,PGHH,-0.78,-47.74,150.73,31.02,168179.0,-101944.0,111576.0,17484.41,-61.63,47.0,X-SC,24.12,57.0,-0.61,0.73,5.04,X40,ATH,FMCG


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
1,ABBOTINDIA,-0.49,-11.19,29.90,15.37,47496.0,-20005.0,158850.0,34390.26,-27.79,54.0,X-MC,1.14,58.0,-0.42,1.04,7.60,X40,ATH,PHARMA
17,CIPLA,0.31,5.87,17.02,23.89,54174.0,17643.0,318298.0,1648.14,20.33,67.0,X-LC,1.20,18.0,0.33,2.08,19.22,X40N,ATH,PHARMA
11,BERGEPAINT,-5.50,-12.06,42.50,25.32,96335.0,-31079.0,226670.0,680.00,-23.45,52.0,X-MC,2.04,68.0,-0.32,1.48,18.86,XY24,NTT,PAINTS
55,PAGEIND,0.07,-7.29,43.30,32.85,107737.0,-19564.0,248815.0,50936.07,-31.08,50.0,X-MC,2.11,55.0,-0.18,1.63,18.81,X40,ATH,APPARELS
77,UNITDSPR,1.13,3.59,16.58,20.77,40215.0,8410.0,242554.0,1644.00,-11.76,58.0,X-MC,3.52,64.0,0.21,1.59,16.59,X40N,NTT,BREWERIES


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
6,AWL,-0.74,-42.65,191.20,67.02,390731.0,-151952.0,204357.0,485.00,-57.79,38.0,X-SC,7.91,81.0,-0.39,1.34,1.65,XY24,NTT,FMCG
28,GODREJCP,-1.73,-22.87,71.28,32.10,137187.0,-57075.0,192462.0,1452.16,-24.40,50.0,X-MC,16.52,32.0,-0.42,1.26,4.28,X40N,ATH,FMCG
56,PGHH,-0.78,-47.74,150.73,31.02,168179.0,-101944.0,111576.0,17484.41,-61.63,47.0,X-SC,24.12,57.0,-0.61,0.73,5.04,X40,ATH,FMCG
40,ICICIPRULI,1.09,-29.23,22.14,-13.56,30866.0,-57591.0,139414.0,522.35,-54.58,44.0,X-MC,9.52,78.0,-1.87,0.91,5.07,X40,ATH,INSURANCE
20,DABUR,-1.61,-25.94,94.03,43.70,172750.0,-64350.0,183718.0,735.00,-36.05,51.0,X-MC,12.65,70.0,-0.37,1.20,5.87,XY24,BTT,FMCG


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
56,PGHH,-0.78,-47.74,150.73,31.02,168179.0,-101944.0,111576.0,17484.41,-61.63,47.0,X-SC,24.12,57.0,-0.61,0.73,5.04,X40,ATH,FMCG
40,ICICIPRULI,1.09,-29.23,22.14,-13.56,30866.0,-57591.0,139414.0,522.35,-54.58,44.0,X-MC,9.52,78.0,-1.87,0.91,5.07,X40,ATH,INSURANCE
2,ACC,-0.64,-45.45,220.48,74.81,322465.0,-121872.0,146256.0,3906.00,-45.85,59.0,X-SC,15.49,85.0,-0.38,0.96,8.35,XY24,BTT,CEMENT
37,HONAUT,-0.86,-22.06,93.59,50.88,140666.0,-42538.0,150300.0,58192.80,-39.13,36.0,X-SC,6.99,90.0,-0.30,0.98,14.51,X40N,ATH,ELECTRICAL
1,ABBOTINDIA,-0.49,-11.19,29.90,15.37,47496.0,-20005.0,158850.0,34390.26,-27.79,54.0,X-MC,1.14,58.0,-0.42,1.04,7.60,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
47,ITC,0.91,-19.87,60.63,28.70,212583.0,-86971.0,350624.0,452.00,-27.49,68.0,X-LC,7.50,1.0,-0.41,2.30,10.35,X40,NTT,FMCG
72,TCS,-0.04,-24.86,68.68,26.75,251874.0,-121339.0,366736.0,4208.34,-39.28,63.0,X-LC,6.41,2.0,-0.48,2.40,27.20,X40,ATH,IT
45,INFY,-0.15,-15.67,69.18,42.66,233845.0,-62831.0,338024.0,1972.00,-38.91,62.0,X-LC,8.46,3.0,-0.27,2.21,21.01,X40,NTT,IT
74,TMPV,-0.82,-29.09,105.97,46.05,336164.0,-130146.0,317226.0,600.00,-85.99,59.0,X-LC,17.24,4.0,-0.39,2.08,11.52,XY24,NTT,AUTO
33,HCLTECH,-0.01,-2.12,28.43,25.70,100446.0,-7664.0,353309.0,1837.07,-6.73,66.0,X-LC,6.94,8.0,-0.08,2.31,40.96,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
76,TTKPRESTIG,9.36,-21.67,27.78,0.09,21930.0,-21836.0,78941.0,770.00,-4.67,72.0,M-SC,3.53,250.0,-1.00,0.52,42.08,OX40N,NTT,DURABLES
12,BOSCH-HCIL,0.20,-26.25,35.58,-0.01,33537.0,-33546.0,94259.0,2282.00,733.99,43.0,M-SC,19.87,231.0,-1.00,0.62,60.44,OX40N,BTT,AC
80,VALIANTORG,-6.70,-28.39,143.18,74.15,167755.0,-46441.0,117164.0,838.00,NaN,34.0,NaN,43.91,145.0,-0.28,0.77,74.36,XR,NTT,CHEMICALS
70,TANLA,-0.63,-26.36,192.19,115.17,383208.0,-71371.0,199390.0,1903.93,NaN,74.0,NaN,14.19,147.0,-0.19,1.31,76.26,AR,ATH,IT
85,ZYDUSLIFE,0.34,26.23,7.35,35.51,19443.0,54963.0,264537.0,1285.01,24.82,56.0,H-LC,11.85,120.0,2.83,1.73,39.21,AR,ATH,PHARMA


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
70,TANLA,-0.63,-26.36,192.19,115.17,383208.0,-71371.0,199390.0,1903.93,NaN,74.0,NaN,14.19,147.0,-0.19,1.31,76.26,AR,ATH,IT
71,TATAELXSI,-1.66,-44.44,137.65,32.03,100819.0,-58592.0,73243.0,9161.00,-36.88,63.0,H-SC,13.52,158.0,-0.58,0.48,28.67,OX40N,NTT,IT
84,WIPRO,-0.04,-8.86,111.77,93.01,254925.0,-22160.0,228080.0,420.00,-23.12,69.0,M-LC,7.84,98.0,-0.09,1.49,26.49,XR,NTT,IT
13,BSOFT,2.44,-38.66,139.96,47.20,120756.0,-54374.0,86279.0,811.90,NaN,64.0,NaN,11.06,172.0,-0.45,0.57,27.33,XR,ATH,IT
33,HCLTECH,-0.01,-2.12,28.43,25.70,100446.0,-7664.0,353309.0,1837.07,-6.73,66.0,X-LC,6.94,8.0,-0.08,2.31,40.96,X40,ATH,IT


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.28
1,25,50.04
2,50,80.57


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    43.21
MC    20.48
SC    20.05
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.59
XY24     22.29
X40N     18.76
XR        9.71
XY25      9.46
OX40N     6.51
AR        4.93
SR        0.74
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    39.99
X-MC    18.09
H-SC     8.13
X-SC     6.99
M-SC     4.93
H-LC     1.73
M-LC     1.49
M-MC     1.06
H-MC     0.98
L-MC     0.35
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,17.71,-26.87,72.42
IT,12.96,-29.58,97.10
FINANCE,10.60,-24.52,77.71
AUTO,8.67,-13.22,53.61
PHARMA,6.18,5.96,19.95
INSURANCE,6.01,-7.65,36.93
MISC,5.38,-50.26,110.65
PAINTS,5.35,-26.09,48.22
BANKS,5.07,-16.18,74.44


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,4030299.0,19
X40,2177685.0,17
XR,1573110.0,13
X40N,1473016.0,12
AR,1040545.0,7
OX40N,795360.0,10
XY25,770185.0,6
SR,300327.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2835090.0,19
X-MC,1642560.0,14
X-SC,1207148.0,6
H-SC,909111.0,8
M-SC,410324.0,6
H-MC,264711.0,1
M-LC,254925.0,1
M-MC,83180.0,1
L-MC,49924.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,,sum,count
Conviction,Criteria,,
X-LC,X40,1266890.0,8
X-SC,XY24,713196.0,2
X-LC,XY24,557323.0,2
H-SC,XY24,556472.0,3
X-MC,X40N,545365.0,4
X-LC,X40N,540216.0,5
X-MC,X40,516664.0,6
X-LC,XY25,375559.0,3
X-MC,XY25,311446.0,2


In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 35.0 seconds
